In [2]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

C:\Users\PMLS\AppData\Local\Temp\ipykernel_8732\1416302830.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
d:\CampusX\Langchain\Components\4_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

True

In [4]:
yt = YouTubeTranscriptApi()
try:
    transcript_list = yt.fetch(video_id="b6e8CCPp2Kc", languages=["en"])
    print(transcript_list)
except TranscriptsDisabled:
    print("Transcript not available")

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='you think about it human intelligence', start=1.56, duration=4.48), FetchedTranscriptSnippet(text='has created all of modern', start=4.16, duration=4.68), FetchedTranscriptSnippet(text='civilization in a way that shows the', start=6.04, duration=5.12), FetchedTranscriptSnippet(text='power of intelligence and how broad it', start=8.84, duration=4.7), FetchedTranscriptSnippet(text='can be', start=11.16, duration=4.08), FetchedTranscriptSnippet(text='[Music]', start=13.54, duration=4.98), FetchedTranscriptSnippet(text='used our mission is solving intelligence', start=15.24, duration=6.84), FetchedTranscriptSnippet(text='to advance science and benefit', start=18.52, duration=3.56), FetchedTranscriptSnippet(text='Humanity real research means doing the', start=22.76, duration=5.439), FetchedTranscriptSnippet(text='unknown we tried to create this unique', start=25.48, duration=4.84), FetchedTranscriptSnippet(text='environment that is 

In [5]:
transcript_text = "".join(chunk.text + " " for chunk in transcript_list.snippets)
print(transcript_text)

you think about it human intelligence has created all of modern civilization in a way that shows the power of intelligence and how broad it can be [Music] used our mission is solving intelligence to advance science and benefit Humanity real research means doing the unknown we tried to create this unique environment that is geared completely towards using artificial intelligence for fast-paced Innovation all right everyone welcome to Deep Mind we are embarking on what will turn out to be the greatest Adventure in human scientific history when we started deep mind we looked at machine learning in neuroscience and there was a good chance that this was going to open up the possibility for full artificial general intelligence our goal is to have an algorithm that can do everything end to end by itself we're going to be pushing the balance in ways that we don't even know as we're doing the research and the development of AGI we need a system which can do any cognitive task that a person can 

In [6]:
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
spliter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs =spliter.create_documents([transcript_text])
print(docs)

[Document(metadata={}, page_content="you think about it human intelligence has created all of modern civilization in a way that shows the power of intelligence and how broad it can be [Music] used our mission is solving intelligence to advance science and benefit Humanity real research means doing the unknown we tried to create this unique environment that is geared completely towards using artificial intelligence for fast-paced Innovation all right everyone welcome to Deep Mind we are embarking on what will turn out to be the greatest Adventure in human scientific history when we started deep mind we looked at machine learning in neuroscience and there was a good chance that this was going to open up the possibility for full artificial general intelligence our goal is to have an algorithm that can do everything end to end by itself we're going to be pushing the balance in ways that we don't even know as we're doing the research and the development of AGI we need a system which can do 

In [7]:
vector_store = FAISS.from_documents(documents=docs, embedding = embedding_model)
print(vector_store)

In [8]:
retriever = vector_store.as_retriever(search_type = "similarity", search_kwargs={"k":4})
result = retriever.invoke("What is DeepMind")
print(result)

[Document(id='c026168f-c599-4a45-ae5e-64ada96c3971', metadata={}, page_content="that are learning to play Atari games and they actually learn more complex behaviors really quickly we're going up the games ladder and actually tackling more complex challenges Deep Mind put its computer program to the test against one of the brightest Minds in the world and [Applause] one star is just playing so smartly that is that the real purpose of working on games for us was always one step on the way to making more General systems there's so much we don't know but how do we start thinking about the downstream impact of our research how do we Pioneer responsibly as an organization we take that job extremely seriously by doing rigorous research in ethics we actually have an opportunity to rethink and say what could AI do for society in a positive way these are metrics from the Renewables project the more people who are using these algorithms in AI the greater the impact we're going to have on trying t

In [9]:
context_text = "".join(doc.page_content+"" for doc in result)
print(context_text)

that are learning to play Atari games and they actually learn more complex behaviors really quickly we're going up the games ladder and actually tackling more complex challenges Deep Mind put its computer program to the test against one of the brightest Minds in the world and [Applause] one star is just playing so smartly that is that the real purpose of working on games for us was always one step on the way to making more General systems there's so much we don't know but how do we start thinking about the downstream impact of our research how do we Pioneer responsibly as an organization we take that job extremely seriously by doing rigorous research in ethics we actually have an opportunity to rethink and say what could AI do for society in a positive way these are metrics from the Renewables project the more people who are using these algorithms in AI the greater the impact we're going to have on trying to stop climate change we've already used them in many domains for exampleyou thi

In [10]:
prompt_template = PromptTemplate(
    template="You are a professional AI Tutor specifically focusing on ML an DL. the question is {question} and the context is {context}. Answer only related to the given text. If not related simply say I do not know",
    input_variables=["question", "context"]
)

prompt = prompt_template.invoke({"question":"What is deep mind", "context":context_text})
print(prompt)

text="You are a professional AI Tutor specifically focusing on ML an DL. the question is What is deep mind and the context is that are learning to play Atari games and they actually learn more complex behaviors really quickly we're going up the games ladder and actually tackling more complex challenges Deep Mind put its computer program to the test against one of the brightest Minds in the world and [Applause] one star is just playing so smartly that is that the real purpose of working on games for us was always one step on the way to making more General systems there's so much we don't know but how do we start thinking about the downstream impact of our research how do we Pioneer responsibly as an organization we take that job extremely seriously by doing rigorous research in ethics we actually have an opportunity to rethink and say what could AI do for society in a positive way these are metrics from the Renewables project the more people who are using these algorithms in AI the grea

In [13]:
llm = ChatGoogleGenerativeAI(model="gemini-flash-latest")
show_result = llm.invoke(input=prompt)
print(show_result)

content=[{'type': 'text', 'text': 'Based on the provided text, **DeepMind** is an international research organization and fast-paced innovative environment that combines machine learning and neuroscience with the following key attributes:\n\n* **Mission:** Its mission is **"solving intelligence to advance science and benefit Humanity."**\n* **Core Goal:** To develop **full Artificial General Intelligence (AGI)**—creating an end-to-end system that can perform any cognitive task that a person can do and potentially scale far beyond that.\n* **Approach:** They pioneer responsibly through rigorous research in ethics, using games and simulations (such as Atari games) as a proving ground to build more general AI systems.\n* **Impact and Applications:** They apply their AI systems to tackle complex, real-world scientific challenges, including:\n  * Solving the 50-year-old biological Grand Challenge of protein folding (AlphaFold).\n  * Addressing climate change through renewable energy project